# Phase 1 — Conformité : vérifier le droit de crawler
**Durée estimée : 30 min**

---

## Objectif
Avant d'écrire une seule ligne de code de scraping, un chef de projet IA doit **prouver que la collecte est autorisée**.  
Dans ce notebook, vous allez analyser les fichiers `robots.txt` de 3 sites et produire un tableau de décision argumenté.

## Ce que vous rendez à la fin
- Un tableau comparatif (3 lignes × 4 colonnes)
- Une décision argumentée : quel site scraper ? pourquoi ?

---
> **Rappel juridique rapide**  
> Un fichier `robots.txt` n'a **pas de valeur légale contraignante** (c'est une convention technique), mais l'ignorer délibérément peut être retenu contre vous dans une procédure (CJUE, Ryanair/PR Aviation, 2015). Les CGU, elles, ont une valeur contractuelle.

In [68]:
# --- Installation (décommentez si besoin) ---
# !pip install requests pandas
# !pip install protego    # parser robots.txt RFC 9309 — utilisé par Scrapy, pure Python

In [69]:
import requests
import pandas as pd
from urllib.robotparser import RobotFileParser
from IPython.display import display, Markdown

print("Imports OK ✓")

Imports OK ✓


---
## Étape 1 — Récupérer et afficher les robots.txt

Exécutez la cellule ci-dessous pour télécharger et afficher le contenu brut des 3 fichiers.  
**Lisez-les attentivement** avant de répondre aux questions.

In [70]:
SITES = {
    "data.gouv.fr": "https://www.data.gouv.fr/robots.txt",
    "lemonde.fr":   "https://www.lemonde.fr/robots.txt",
    "wikipedia.fr": "https://fr.wikipedia.org/robots.txt",
    "indeed.fr":    "https://fr.indeed.com/robots.txt",   # ← site d'offres d'emploi
}

robots_raw = {}

for nom, url in SITES.items():
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        robots_raw[nom] = r.text
        print(f"✅ {nom} — {len(r.text)} caractères récupérés")
    except Exception as e:
        robots_raw[nom] = ""
        print(f"❌ {nom} — Erreur : {e}")

✅ data.gouv.fr — 2170 caractères récupérés
✅ lemonde.fr — 11995 caractères récupérés
✅ wikipedia.fr — 16761 caractères récupérés
✅ indeed.fr — 14534 caractères récupérés


In [71]:
# Affiche le robots.txt d'un site — changez le nom pour voir les autres
SITE_A_AFFICHER = "data.gouv.fr"  # <-- modifiez : "lemonde.fr", "wikipedia.fr" ou "indeed.fr"

print(f"=== robots.txt de {SITE_A_AFFICHER} ===")
print(robots_raw[SITE_A_AFFICHER][:3000])
print("[...]") if len(robots_raw[SITE_A_AFFICHER]) > 3000 else None

=== robots.txt de data.gouv.fr ===
User-agent: *
Disallow: /admin
Disallow: /en/admin
Disallow: /fr/admin
Disallow: /es/admin
Disallow: /datasets/search?*
Disallow: /en/datasets/search?*
Disallow: /fr/datasets/search?*
Disallow: /es/datasets/search?*
Disallow: /reuses/search?*
Disallow: /en/reuses/search?*
Disallow: /fr/reuses/search?*
Disallow: /es/reuses/search?*
Disallow: /dataservices/search?*
Disallow: /en/dataservices/search?*
Disallow: /fr/dataservices/search?*
Disallow: /es/dataservices/search?*
Disallow: /organizations?*
Disallow: /en/organizations?*
Disallow: /fr/organizations?*
Disallow: /es/organizations?*
Disallow: /datasets.csv
Disallow: /en/datasets.csv
Disallow: /fr/datasets.csv
Disallow: /es/datasets.csv
Disallow: /resources.csv
Disallow: /en/resources.csv
Disallow: /fr/resources.csv
Disallow: /es/resources.csv
Disallow: /reuses.csv
Disallow: /en/reuses.csv
Disallow: /fr/reuses.csv
Disallow: /es/reuses.csv
Disallow: /dataservices.csv
Disallow: /en/dataservices.csv
Disa

In [72]:
# Affiche le robots.txt d'un site — changez le nom pour voir les autres
SITE_A_AFFICHER = "indeed.fr"  # <-- modifiez : "lemonde.fr", "wikipedia.fr" ou "indeed.fr"

print(f"=== robots.txt de {SITE_A_AFFICHER} ===")
print(robots_raw[SITE_A_AFFICHER][:3000])
print("[...]") if len(robots_raw[SITE_A_AFFICHER]) > 3000 else None

=== robots.txt de indeed.fr ===
User-agent: *
Allow: /
Allow: /hire/*?*isid=
Allow: /personeel/*?*isid=
Allow: /reclutamiento/*?*isid=
Allow: /recruiting/*?*isid=
Allow: /recrutement/*?*isid=
Disallow: /*rt=nc
Disallow: /*&alid=
Disallow: /*&calert=
Disallow: /*&iafilter=
Disallow: /*&mna=
Disallow: /*?rss
Disallow: /addlLoc/
Disallow: /ads/
Disallow: /advanced_search
Disallow: /alert
Disallow: /api/fetch/mc-anon
Disallow: /api/getrecjobs
Disallow: /applystart
Disallow: /cmp/_/
Disallow: /cmp/_c/claim/*
Disallow: /cmp/_rpc/
Disallow: /cmp/addlink
Disallow: /cmp/addvideo
Disallow: /cmp/Login*
Disallow: /cmp/*/analytics
Disallow: /cmp/*/company-questions
Disallow: /cmp/*/people
Disallow: /cmp/*/write-review
Disallow: /community/
Disallow: /company/*
Disallow: /conversion/
Disallow: /cookiemigrator/
Disallow: /create-resume/lp/
Disallow: /cdn-cgi/
Disallow: /%E4%BB%95%E4%BA%8B?
Disallow: /%E5%B7%A5%E4%BD%9C/
Disallow: /%E5%B7%A5%E4%BD%9C/CN/
Disallow: /%E5%B7%A5%E4%BD%9C/title/
Disallow: 

---
## Étape 2 — Tester des URLs spécifiques

La bibliothèque `urllib.robotparser` intégrée à Python peut lire un robots.txt et répondre automatiquement : "cette URL est-elle autorisée pour User-agent: *  ?"

Nous allons tester 2 URLs par site :
- une **URL "safe"** (page de contenu standard)
- une **URL "risquée"** (page de recherche, paramètre `?q=`, etc.)

In [73]:
TESTS = {
    "data.gouv.fr": (
        "https://www.data.gouv.fr/robots.txt",
        [
            "https://www.data.gouv.fr/fr/datasets/",            # page de liste — safe
            "https://www.data.gouv.fr/fr/search/?q=education",  # recherche — risquée
        ]
    ),
    "lemonde.fr": (
        "https://www.lemonde.fr/robots.txt",
        [
            "https://www.lemonde.fr/economie/",                 # rubrique — safe
            "https://www.lemonde.fr/recherche/?keywords=ia",    # recherche — risquée
        ]
    ),
    "wikipedia.fr": (
        "https://fr.wikipedia.org/robots.txt",
        [
            "https://fr.wikipedia.org/wiki/Intelligence_artificielle",  # article — safe
            "https://fr.wikipedia.org/w/index.php?search=IA",          # recherche interne — risquée
        ]
    ),
    "indeed.fr": (
        "https://fr.indeed.com/robots.txt",
        [
            "https://fr.indeed.com/emplois?q=data+scientist&l=Paris",              # recherche — safe ?
            "https://fr.indeed.com/viewjob?jk=c44fa6bcc73bd5d3&from=shareddesktop_copy",  # fiche offre — risquée
        ]
    ),
}

def lire_robots(robots_url: str) -> RobotFileParser:
    """Charge un robots.txt en gérant le BOM UTF-8 (ex. Wikipedia)."""
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        r = requests.get(robots_url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        contenu_propre = r.text.lstrip("﻿")  # strip BOM si présent
        rp.parse(contenu_propre.splitlines())
    except Exception as e:
        print(f"  ⚠️  Impossible de lire {robots_url} : {e}")
    return rp

resultats = []

for site, (robots_url, urls) in TESTS.items():
    rp = lire_robots(robots_url)
    for url in urls:
        autorisee = rp.can_fetch("*", url)
        delay     = rp.crawl_delay("*")
        resultats.append({
            "Site":        site,
            "URL testée":  url,
            "Autorisée ?": "✅ Oui" if autorisee else "❌ Non",
            "Crawl-delay": str(delay) + "s" if delay else "Non défini",
        })

df_tests = pd.DataFrame(resultats)
display(df_tests)

# Détection BOM
print("\n📌 BOM UTF-8 détecté sur :")
for nom, url in [
    ("data.gouv.fr", "https://www.data.gouv.fr/robots.txt"),
    ("lemonde.fr",   "https://www.lemonde.fr/robots.txt"),
    ("wikipedia.fr", "https://fr.wikipedia.org/robots.txt"),
    ("indeed.fr",    "https://fr.indeed.com/robots.txt"),
]:
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        bom = r.text.startswith("﻿")
        print(f"  {nom} : {'⚠️  OUI → parsing manuel nécessaire' if bom else 'Non'}")
    except:
        pass

# ⚠️  Vérification manuelle pour Indeed — le parser Python donne un faux positif
print()
print("⚠️  Vérification manuelle Indeed — 'Disallow: /viewjob?' dans le bloc User-agent: * ?")
viewjob_disallow = "Disallow: /viewjob?" in robots_raw.get("indeed.fr", "")
print(f"   Trouvé dans robots.txt : {'OUI → /viewjob? est explicitement interdit aux bots anonymes' if viewjob_disallow else 'Non'}")

,Site,URL testée,Autorisée ?,Crawl-delay
0,data.gouv.fr,https://www.data.gouv.fr/fr/datasets/,✅ Oui,Non défini
1,data.gouv.fr,https://www.data.gouv.fr/fr/search/?q=education,✅ Oui,Non défini
2,lemonde.fr,https://www.lemonde.fr/economie/,✅ Oui,Non défini
3,lemonde.fr,https://www.lemonde.fr/recherche/?keywords=ia,✅ Oui,Non défini
4,wikipedia.fr,https://fr.wikipedia.org/wiki/Intelligence_art...,✅ Oui,Non défini
5,wikipedia.fr,https://fr.wikipedia.org/w/index.php?search=IA,❌ Non,Non défini
6,indeed.fr,https://fr.indeed.com/emplois?q=data+scientist...,✅ Oui,Non défini
7,indeed.fr,https://fr.indeed.com/viewjob?jk=c44fa6bcc73bd...,✅ Oui,Non défini



📌 BOM UTF-8 détecté sur :
  data.gouv.fr : Non
  lemonde.fr : Non
  wikipedia.fr : ⚠️  OUI → parsing manuel nécessaire
  indeed.fr : Non

⚠️  Vérification manuelle Indeed — 'Disallow: /viewjob?' dans le bloc User-agent: * ?
   Trouvé dans robots.txt : OUI → /viewjob? est explicitement interdit aux bots anonymes


---
## Comment lire ce tableau ?

| Ligne | Résultat | À retenir |
|-------|----------|-----------|
| data.gouv.fr `/fr/search/?q=` | ✅ Oui | `/fr/search/` n'est pas dans les Disallow — deux chemins distincts. |
| lemonde.fr `/recherche/` | ✅ Oui | Autorisé par robots.txt, mais Le Monde utilise du JS + tokens anti-bot. |
| wikipedia `/wiki/…` | ✅ Oui | Autorisé. Sans le fix BOM, affichait ❌ à tort (voir ci-dessous). |
| wikipedia `/w/index.php?search=` | ❌ Non | Correct — `Disallow: /w/` dans `User-agent: *`. |
| indeed `/viewjob?jk=…` | ✅ Oui | **⚠️ FAUX POSITIF** — `urllib.robotparser` (spec 1994) donne une réponse incorrecte. |

**Règle d'or :** robots.txt dit ce que le site *tolère*, les CGU disent ce que vous avez *le droit* de faire. Toujours vérifier les deux.

### ⚠️ Indeed `/viewjob?` : faux positif de la stdlib Python

`urllib.robotparser` implémente une spec de 1994, pas le standard actuel (RFC 9309). Il retourne ✅ Oui alors que le robots.txt interdit explicitement `/viewjob?` aux bots anonymes.  
La cellule ci-dessous corrige ça avec une implémentation RFC 9309.

### BOM UTF-8 (Wikipedia)

Wikipedia sert son robots.txt avec un caractère invisible `U+FEFF` en tête — `urllib.robotparser` ne le strip pas → toutes les URLs revenaient ❌ à tort. Corrigé par `.lstrip("﻿")` dans `lire_robots()`.

---
## ⚠️ Attention à la bibliothèque utilisée

`urllib.robotparser` (incluse dans Python) implémente une spécification de **1994** — elle donne un faux positif sur Indeed `/viewjob?` et ne gère pas le BOM UTF-8 de Wikipedia sans correctif manuel.

**La solution : [`protego`](https://pypi.org/project/protego/)** — la bibliothèque utilisée par le framework **Scrapy**.
- Conforme **RFC 9309** (standard actuel, identique au comportement de Google)
- Gère le **BOM UTF-8** nativement (Wikipedia fonctionne sans `.lstrip()`)
- Pure Python — `pip install protego`, aucune dépendance C++

```bash
pip install protego
```

La cellule ci-dessous compare les deux bibliothèques sur le cas Indeed.

In [74]:
from protego import Protego

URL_OFFRE  = "https://fr.indeed.com/viewjob?jk=c44fa6bcc73bd5d3"
contenu    = robots_raw.get("indeed.fr", "")

rp_std     = lire_robots("https://fr.indeed.com/robots.txt")   # urllib.robotparser (1994)
rp_protego = Protego.parse(contenu)                            # protego RFC 9309

agents_test = [
    ("MonScraper/1.0", "Bot anonyme (votre scraper)"),
    ("Googlebot",      "Googlebot (Google Search)"),
    ("Claude-User",    "Claude-User (Anthropic)"),
    ("TP-Scraping",    "User-agent custom (inconnu d'Indeed)"),
]

print("=" * 68)
print("  stdlib Python (1994)  vs  protego RFC 9309  —  /viewjob?")
print("=" * 68)
print(f"\n  {'Agent':<34} {'stdlib':>8}   {'protego':>8}")
print("  " + "─" * 62)

for agent, label in agents_test:
    std = rp_std.can_fetch(agent, URL_OFFRE)
    pro = rp_protego.can_fetch(URL_OFFRE, agent)   # ⚠️ protego : url en premier, agent en second
    bug = "  ← BUG" if std and not pro else ""
    print(f"  {label:<34}  {'✅ Oui' if std else '❌ Non':>6}     {'✅ Oui' if pro else '❌ Non':>6}{bug}")

print("  " + "─" * 62)
print()
print("✅ protego donne la réponse correcte : /viewjob? est interdit aux bots anonymes.")
print()

# Bonus — protego gère aussi le BOM Wikipedia nativement
print("── Bonus : Wikipedia (BOM UTF-8) ──")
contenu_wiki = robots_raw.get("wikipedia.fr", "")
rp_wiki = Protego.parse(contenu_wiki)
print(f"  /wiki/Intelligence_artificielle : {'✅ Oui' if rp_wiki.can_fetch('https://fr.wikipedia.org/wiki/Intelligence_artificielle', 'MonScraper/1.0') else '❌ Non'}  (sans aucun fix BOM)")
print(f"  /w/index.php?search=IA          : {'✅ Oui' if rp_wiki.can_fetch('https://fr.wikipedia.org/w/index.php?search=IA', 'MonScraper/1.0') else '❌ Non'}")

  stdlib Python (1994)  vs  protego RFC 9309  —  /viewjob?

  Agent                                stdlib    protego
  ──────────────────────────────────────────────────────────────
  Bot anonyme (votre scraper)          ✅ Oui      ❌ Non  ← BUG
  Googlebot (Google Search)            ✅ Oui      ✅ Oui
  Claude-User (Anthropic)              ✅ Oui      ✅ Oui
  User-agent custom (inconnu d'Indeed)   ✅ Oui      ❌ Non  ← BUG
  ──────────────────────────────────────────────────────────────

✅ protego donne la réponse correcte : /viewjob? est interdit aux bots anonymes.

── Bonus : Wikipedia (BOM UTF-8) ──
  /wiki/Intelligence_artificielle : ✅ Oui  (sans aucun fix BOM)
  /w/index.php?search=IA          : ❌ Non


---
## Étape 3 — Tableau comparatif (à compléter)

Complétez le dictionnaire `votre_analyse` ci-dessous avec vos conclusions.  
**Soyez précis** : citez des règles réelles trouvées dans les robots.txt.

In [75]:
from IPython.display import HTML

# ✏️  COMPLÉTEZ CE DICTIONNAIRE avec vos observations
# Les lignes Wikipedia et Indeed sont pré-remplies comme exemples de correction.
# ⚠️  Deux dimensions à distinguer :
#   - "Risque crawling"  : ai-je le droit d'envoyer des requêtes ? (robots.txt, CGU scraping, anti-bot)
#   - "Risque données"   : ai-je le droit de réutiliser ce que je collecte ? (licence, copyright)
votre_analyse = [
    {
        "Site":             "data.gouv.fr",
        "Pages interdites": "À compléter<br><em>ex: /fr/datasets/search?* est bloqué, mais pas /fr/search/</em>",
        "Crawl-delay":      "À compléter",
        "Risque crawling":  "À compléter<br><em>robots.txt permissif ? anti-bot ? CGU ?</em>",
        "Risque données":   "À compléter<br><em>licence ? copyright ? réutilisation autorisée ?</em>",
    },
    {
        "Site":             "lemonde.fr",
        "Pages interdites": "À compléter",
        "Crawl-delay":      "À compléter",
        "Risque crawling":  "À compléter",
        "Risque données":   "À compléter",
    },
    {
        "Site":             "fr.wikipedia.org",
        "Pages interdites": "/w/index.php?search= (recherche interne)<br>/wiki/Special:<br><em>Les articles /wiki/Titre sont autorisés</em>",
        "Crawl-delay":      "Non défini<br>→ appliquer 1–2 s par politesse",
        "Risque crawling":  "<strong>Faible</strong><br>robots.txt explicitement permissif pour /wiki/<br>Pas d'anti-bot — une API publique est même disponible",
        "Risque données":   "<strong>Faible</strong><br>Licence CC BY-SA 4.0 — réutilisation libre<br>avec attribution et partage à l'identique",
    },
    {
        "Site":             "fr.indeed.com",
        "Pages interdites": "/viewjob? (fiches offres)<br>/emploi/, /job/, /my/, /graphql<br><em>+ ~150 autres chemins</em>",
        "Crawl-delay":      "Non défini<br><em>détection anti-bot active côté serveur</em>",
        "Risque crawling":  "<strong>Élevé</strong><br>/viewjob? interdit aux bots anonymes (RFC 9309)<br>CGU interdisent explicitement le scraping<br>Asymétrie User-agent: * vs Googlebot",
        "Risque données":   "<strong>Élevé</strong><br>Données propriétaires Indeed<br>CGU interdisent l'extraction et la réutilisation commerciale",
    },
]

df_analyse = pd.DataFrame(votre_analyse)
display(HTML(
    "<style>table td { vertical-align: top; padding: 6px 10px; } "
    "table th { background: #f0f0f0; padding: 6px 10px; }</style>"
    + df_analyse.to_html(index=False, escape=False)
))

Site,Pages interdites,Crawl-delay,Risque crawling,Risque données
data.gouv.fr,"À compléterex: /fr/datasets/search?* est bloqué, mais pas /fr/search/",À compléter,À compléterrobots.txt permissif ? anti-bot ? CGU ?,À compléterlicence ? copyright ? réutilisation autorisée ?
lemonde.fr,À compléter,À compléter,À compléter,À compléter
fr.wikipedia.org,/w/index.php?search= (recherche interne)/wiki/Special:Les articles /wiki/Titre sont autorisés,Non défini→ appliquer 1–2 s par politesse,Faiblerobots.txt explicitement permissif pour /wiki/Pas d'anti-bot — une API publique est même disponible,FaibleLicence CC BY-SA 4.0 — réutilisation libreavec attribution et partage à l'identique
fr.indeed.com,"/viewjob? (fiches offres)/emploi/, /job/, /my/, /graphql+ ~150 autres chemins",Non définidétection anti-bot active côté serveur,Élevé/viewjob? interdit aux bots anonymes (RFC 9309)CGU interdisent explicitement le scrapingAsymétrie User-agent: * vs Googlebot,ÉlevéDonnées propriétaires IndeedCGU interdisent l'extraction et la réutilisation commerciale


---
## Étape 4 — Décision Chef de Projet

Répondez aux 3 questions ci-dessous dans la cellule Markdown suivante.

### Votre décision (à rédiger ici)

**1. Classement du moins risqué au plus risqué :**

| Rang | Site | Argument principal |
|------|------|--------------------|
| 1 (le moins risqué) | *à compléter* | *à compléter* |
| 2 | *à compléter* | *à compléter* |
| 3 (le plus risqué) | *à compléter* | *à compléter* |

**2. Site retenu pour la phase Collecte :**  
*à compléter — ex: Wikipedia, parce que...*

**3. Pause que vous appliquerez entre les requêtes :**  
*à compléter — ex: 2 secondes car le robots.txt ne précise pas de crawl-delay*

---
## ✅ Fin de la Phase 1

Avant de passer au notebook `02_collecte.ipynb`, vérifiez :
- [ ] Votre tableau comparatif est rempli avec des arguments tirés du robots.txt réel
- [ ] Vous avez testé au moins 2 URLs par site
- [ ] Vous avez choisi un site et justifié votre choix

**Passez maintenant à `02_collecte.ipynb` →**